In [42]:
# %pip install pydriller

In [43]:
import re
from pydriller import Repository

In [44]:
ISSUE_IDS = ["TIKA-6", "TIKA-172", "TIKA-605", "TIKA-1722", "TIKA-3523"]
REPO_URL  = "https://github.com/apache/tika.git"
TRACKED_CHANGES = {"ADD", "MODIFY", "DELETE"}

In [45]:
patterns = {i: re.compile(rf"\b{re.escape(i)}\b") for i in ISSUE_IDS}
issue_to_commits = {i: [] for i in ISSUE_IDS}

for commit in Repository(REPO_URL).traverse_commits():
    msg = commit.msg or ""
    matched = [i for i, p in patterns.items() if p.search(msg)]
    if not matched:
        continue

    files = [
        (m.new_path or m.old_path)
        for m in commit.modified_files
        if m.change_type.name in TRACKED_CHANGES
    ]
    info = {
        "hash":      commit.hash,
        "author":    commit.author.name,
        "date":      commit.author_date.strftime("%Y-%m-%d"),
        "subject":   msg.splitlines()[0],
        "files":     files,
        "dmm_size":  commit.dmm_unit_size,
        "dmm_cmplx": commit.dmm_unit_complexity,
        "dmm_iface": commit.dmm_unit_interfacing,
    }
    for issue_id in matched:
        issue_to_commits[issue_id].append(info)

In [46]:
unique_commits = {}
for commits in issue_to_commits.values():
    for c in commits:
        unique_commits[c["hash"]] = c

In [47]:
total_commits = len(unique_commits)

all_files = set()
for c in unique_commits.values():
    all_files.update(c["files"])
avg_files = len(all_files) / total_commits

total_dmm = 0.0
for c in unique_commits.values():
    parts = [p for p in (c["dmm_size"], c["dmm_cmplx"], c["dmm_iface"]) if p is not None]
    if parts:
        total_dmm += sum(parts) / len(parts)
avg_dmm = total_dmm / total_commits

print(f"Total commits analyzed: {total_commits}")
print(f"Average number of files changed: {avg_files:.3f}")
print(f"Average DMM Metrics: {avg_dmm:.6f}")

Total commits analyzed: 12
Average number of files changed: 2.667
Average DMM Metrics: 0.423034
